# Optimisation des hyper-paramètres

Les hyper-paramètres du LightGBM sont cherchés par un Tree of Parzen Estimators (Hyperopt).
Les matrices lues sont `data/X.csv` et `data/y.csv`, ainsi que `data/X_train.csv`, `data/y_train.csv`, `data/X_test.csv` et `data/y_test.csv`.
L'objectif minimisé est l'opposé du F1, estimé sur 4 plis, pour 20 évaluations.
Le modèle ajusté est enregistré dans `data/model.pkl`.
La courbe de calibration compare, par tranche, la fréquence d'achats observée à la probabilité prédite.


## Optimisation bayésienne

Les paramètres du modèle sont estimés pendant l'entraînement. Les hyper-paramètres sont fixés avant.
Une recherche par grille parcourt une liste de points. L'approche bayésienne choisit le point suivant à partir des scores déjà obtenus.
SMBO s'arrête au bout de T itérations. Un substitut approche le score en fonction des hyper-paramètres. La fonction d'acquisition est l'amélioration espérée.
À chaque itération, le jeu qui maximise l'acquisition est entraîné, son score est archivé, puis le substitut est mis à jour.
Les substituts usuels sont le processus gaussien, la forêt aléatoire et le Tree of Parzen Estimators (TPE).
Le TPE estime deux densités : `l` si la perte est inférieure au seuil `y*`, `g` si elle lui est supérieure ou égale. L'amélioration espérée augmente avec le rapport `l(u) / g(u)`.


## Recherche TPE

Hyperopt minimise l'objectif. La métrique renvoyée est l'opposé du F1.
`hp.quniform` renvoie des flottants. Les clés de `override_schemas` sont converties en entiers avant l'instanciation du LightGBM.
Le critère est le F1. Il pénalise un modèle qui prédit toujours la classe positive.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()


In [ ]:
X = pd.read_csv(os.path.expanduser("data/X.csv"))
y = pd.read_csv(os.path.expanduser("data/y.csv"))

X_train = pd.read_csv(os.path.expanduser("data/X_train.csv"))
X_test = pd.read_csv(os.path.expanduser("data/X_test.csv"))
y_train = pd.read_csv(os.path.expanduser("data/y_train.csv")).values.flatten()
y_test = pd.read_csv(os.path.expanduser("data/y_test.csv")).values.flatten()


In [ ]:
from lightgbm.sklearn import LGBMClassifier
from hyperopt import hp, tpe, fmin

MODEL_SPECS = {
    "name": "LightGBM",
    "class": LGBMClassifier,
    "max_evals": 20,
    "params": {
        "learning_rate": hp.uniform("learning_rate", 0.001, 1),
        "num_iterations": hp.quniform("num_iterations", 100, 1000, 20),
        "max_depth": hp.quniform("max_depth", 4, 12, 6),
        "num_leaves": hp.quniform("num_leaves", 8, 128, 10),
        "colsample_bytree": hp.uniform("colsample_bytree", 0.3, 1),
        "subsample": hp.uniform("subsample", 0.5, 1),
        "min_child_samples": hp.quniform("min_child_samples", 1, 20, 10),
        "reg_alpha": hp.choice("reg_alpha", [0, 1e-1, 1, 2, 5, 10]),
        "reg_lambda": hp.choice("reg_lambda", [0, 1e-1, 1, 2, 5, 10]),
    },
    "override_schemas": {
        "num_leaves": int,
        "min_child_samples": int,
        "max_depth": int,
        "num_iterations": int,
    },
}


In [ ]:
from sklearn.model_selection import RepeatedKFold

def optimize_hyp(instance, training_set, search_space, metric, evals=10):
    def objective(params):
        # quniform renvoie des flottants : cast vers le type déclaré
        for param in set(list(MODEL_SPECS["override_schemas"].keys())).intersection(set(params.keys())):
            cast_instance = MODEL_SPECS["override_schemas"][param]
            params[param] = cast_instance(params[param])

        rep_kfold = RepeatedKFold(n_splits=4, n_repeats=1)
        scores_test = []
        for train_I, test_I in rep_kfold.split(X):
            X_fold_train = X.iloc[train_I, :]
            y_fold_train = y.iloc[train_I]
            X_fold_test = X.iloc[test_I, :]
            y_fold_test = y.iloc[test_I]

            model = LGBMClassifier(**params, objective="binary", verbose=-1)
            model.fit(X_fold_train, y_fold_train)
            scores_test.append(metric(y_fold_test, model.predict(X_fold_test)))

        return np.mean(scores_test)

    return fmin(fn=objective, space=search_space, algo=tpe.suggest, max_evals=evals)


In [ ]:
from sklearn.metrics import f1_score

import warnings
warnings.filterwarnings("ignore")

optimum_params = optimize_hyp(
    MODEL_SPECS["class"],
    training_set=(X_train, y_train),
    search_space=MODEL_SPECS["params"],
    metric=lambda x, y: -f1_score(x, y),  # Hyperopt minimise : opposé du F1
    evals=MODEL_SPECS["max_evals"],
)


In [ ]:
optimum_params


In [ ]:
for param in MODEL_SPECS["override_schemas"]:
    cast_instance = MODEL_SPECS["override_schemas"][param]
    optimum_params[param] = cast_instance(optimum_params[param])

model = LGBMClassifier(**optimum_params)
model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import recall_score, precision_score

print("F1 Score : {:2.1f}%".format(f1_score(y_test, model.predict(X_test)) * 100))
print("Precision : {:2.1f}%".format(precision_score(y_test, model.predict(X_test)) * 100))
print("Recall : {:2.1f}%".format(recall_score(y_test, model.predict(X_test)) * 100))


In [ ]:
import joblib

joblib.dump(model, os.path.expanduser("data/model.pkl"))


## Courbe de calibration

Pour chaque tranche de probabilité prédite, le graphique compare la proportion réelle de positifs à la probabilité moyenne de la tranche.
La diagonale correspond à des fréquences alignées sur les probabilités.


In [ ]:
import matplotlib.ticker as mtick

from sklearn.calibration import calibration_curve

prob_pos = model.predict_proba(X_test)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, prob_pos, n_bins=20)

plt.figure(figsize=(16, 10))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated", alpha=0.6)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model")
plt.ylabel("Fraction of positives")
plt.xlabel("Predicted probabilites")
plt.legend()
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1, 0))
plt.title("Calibration Curve")
plt.show()
